In [0]:
from pyspark.sql import functions as F

T = "workspace.heatcheck."
wbgt = spark.table(T + "silver_wbgt")
risk = spark.table(T + "gold_subzone_risk")
seniors = spark.table(T + "silver_seniors_by_subzone")

s = wbgt.agg(F.count("*").alias("readings"), F.countDistinct("station_id").alias("stations"),
             F.min("reading_time").alias("first"), F.max("reading_time").alias("last")).first()
tot = seniors.agg(F.sum("seniors_65plus").alias("s65"),
                  F.sum("seniors_small_flats").alias("ssf")).first()
fs = (risk.filter("risk_band = 'Check first'")
      .agg(F.count("*").alias("n"), F.sum("seniors_small_flats").alias("ssf")).first())
fn = risk.filter("risk_band IN ('Check first', 'Check next')").agg(F.sum("seniors_small_flats")).first()[0]
elder = spark.table(T + "silver_eldercare_points").count()

print(f"WBGT readings used: {s.readings:,} from {s.stations} stations, {s.first:%d %b %Y} to {s.last:%d %b %Y}")
print(f"Seniors 65+ (June 2026): {tot.s65:,}")
print(f"Seniors 65+ in HDB 1-2 room flats: {tot.ssf:,}")
print(f"Subzones scored: {risk.count()} | Check first: {fs.n}")
print(f"Check first covers {fs.ssf:,} seniors in 1-2 room flats ({fs.ssf / tot.ssf:.0%} of all of them)")
print(f"Check first + Check next cover {fn:,} ({fn / tot.ssf:.0%})")
print(f"Eldercare and AAC locations used: {elder}")